# 03 - Feature Engineering and Label Creation

build features (utilization, spend, chronic conditions, demographics, rx) from the **2008-2009 feature window**, then derive high_cost / high_utilization labels from the **2010 label window**. the same sql runs twice with different bound dates -- once for features, once for the label-window aggregate -- so no 2010 activity ever leaks into a predictor.

In [ ]:
import duckdb
import pandas as pd
import yaml
from pathlib import Path

with open("../configs/data_sources.yaml") as f:
    config = yaml.safe_load(f)
with open("../configs/model_config.yaml") as f:
    model_config = yaml.safe_load(f)

interim_dir = Path("..") / config["interim_dir"]
processed_dir = Path("..") / config["processed_dir"]

# dates in the raw files are plain yyyymmdd integers, not a date type, so
# window boundaries are bound as ints too
FEATURE_START, FEATURE_END = 20080101, 20091231
LABEL_START, LABEL_END = 20100101, 20101231


In [ ]:
# register interim tables + cohort as duckdb views
con = duckdb.connect()
for name in ["beneficiary_summary_2008", "beneficiary_summary_2009", "beneficiary_summary_2010",
             "inpatient_claims", "outpatient_claims", "carrier_claims",
             "prescription_drug_events"]:
    con.execute(f"""
        create view {name} as
        select * from read_parquet('{interim_dir}/{name}.parquet')
    """)

con.execute(f"""
    create view continuously_enrolled as
    select * from read_parquet('{processed_dir}/cohort.parquet')
""")


In [ ]:
# small helper: read a sql file and bind a :window_start/:window_end pair
def run_windowed(path, window_start, window_end):
    sql = Path(path).read_text()
    sql = sql.replace(":window_start", str(window_start)).replace(":window_end", str(window_end))
    return con.execute(sql).df()


In [ ]:
# feature-window aggregates: utilization/spend + rx, computed over 2008-2009
utilization = run_windowed("../sql/features/utilization_features.sql", FEATURE_START, FEATURE_END)
pharmacy = run_windowed("../sql/features/pharmacy_features.sql", FEATURE_START, FEATURE_END)

# chronic conditions + demographics aren't windowed -- they're a snapshot as
# of the most recent feature-window year (2009), not a sum over a date range.
# other-payer amounts are only available as annual beneficiary-level totals
# (not claim-level), so they're summed across the two feature-window years
# directly rather than run through run_windowed
chronic = con.execute(Path("../sql/features/chronic_condition_features.sql").read_text()).df()
demographics = con.execute(Path("../sql/features/demographic_features.sql").read_text()).df()
other_payer = con.execute(Path("../sql/features/other_payer_features.sql").read_text()).df()

features = (
    utilization
    .merge(chronic, on="desynpuf_id", how="left")
    .merge(demographics, on="desynpuf_id", how="left")
    .merge(pharmacy, on="desynpuf_id", how="left")
    .merge(other_payer, on="desynpuf_id", how="left")
)
features.head()


In [ ]:
# label-window aggregates: same utilization sql, bound to 2010 instead, plus
# the 2010 beneficiary summary's own annual medreimb totals for cost
utilization_2010 = run_windowed("../sql/features/utilization_features.sql", LABEL_START, LABEL_END)
utilization_2010 = utilization_2010.rename(columns={"inpatient_admits": "inpatient_admits_2010"})

cost_2010 = con.execute("""
    select
        c.desynpuf_id,
        b.medreimb_ip + b.medreimb_op + b.medreimb_car as total_paid_2010
    from continuously_enrolled c
    join beneficiary_summary_2010 b on c.desynpuf_id = b.desynpuf_id
""").df()

label_source = cost_2010.merge(
    utilization_2010[["desynpuf_id", "inpatient_admits_2010"]], on="desynpuf_id", how="left"
)


In [ ]:
# derive the two binary labels from the 2010 aggregates
cost_threshold = label_source["total_paid_2010"].quantile(model_config["label"]["high_cost_percentile"])
util_threshold = model_config["label"]["high_utilization_admit_threshold"]

label_source["high_cost"] = (label_source["total_paid_2010"] >= cost_threshold).astype(int)
label_source["high_utilization"] = (label_source["inpatient_admits_2010"] >= util_threshold).astype(int)

label_source[["high_cost", "high_utilization"]].mean()


In [ ]:
# join labels onto the feature table, fill any unmatched rows with 0, and save
features_labeled = features.merge(label_source, on="desynpuf_id", how="left")
features_labeled = features_labeled.fillna(0)

out_path = processed_dir / "features_labeled.parquet"
features_labeled.to_parquet(out_path, index=False)
print(f"wrote {out_path} ({len(features_labeled):,} rows, {features_labeled.shape[1]} cols)")
